Database Connection

In [4]:
import os

import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(find_dotenv())

def connect_to_db():
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")

    if not all([database, user, password]):
        print("Σφάλμα: Δεν βρέθηκαν οι απαραίτητες μεταβλητές στο .env!")
        return None

    try:
        connection_uri = f"postgresql://{user}:{password}@{host}:{port}/{database}"
        engine = create_engine(connection_uri)
        with engine.connect() as connection:
            print(f"Επιτυχής σύνδεση στη βάση '{database}' στο host '{host}'!")
        return engine

    except SQLAlchemyError as e:
        print(f"Σφάλμα κατά τη σύνδεση στη ΒΔ: {e}")
        return None


print("Εκκίνηση σύνδεσης με τη βάση...")
engine = connect_to_db()

if engine is None:
    print("Τερματισμός προγράμματος λόγω αποτυχίας σύνδεσης.")
    exit(1)

Εκκίνηση σύνδεσης με τη βάση...
Επιτυχής σύνδεση στη βάση 'thesis_db' στο host 'dell-micro'!


Import data to dataframe

In [5]:
import pandas as pd

query = "SELECT match_id, x_loc, y_loc, distance_to_goal, angle_to_goal, is_goal FROM understat_shots_normalized"
us_shots = pd.read_sql(query, con=engine)


query = "SELECT match_id, x_loc, y_loc, distance_to_goal, angle_to_goal, is_goal FROM statsbomb_shots_normalized"
sb_shots = pd.read_sql(query, con=engine)

Remove overlapping matches

In [6]:
query = "SELECT understat_match_id FROM mapping_overlapping_matches"
overlapping_matches = pd.read_sql_query(query, con=engine)
overlapping_matches.head()

,understat_match_id
0,923
1,662
2,727
3,890
4,644


In [7]:
print(f"Αρχικές εγγραφές us_shots: {len(us_shots)}")
overlapping_matches_list = overlapping_matches['understat_match_id'].unique()

# Αφαίρεση overlapping matches από το us_shots
us_shots = us_shots[~us_shots['match_id'].isin(overlapping_matches_list)].reset_index(drop=True)

print(f"Overlapping matches: {len(overlapping_matches_list)}")
print(f"Εγγραφές μετά την αφαίρεση: {len(us_shots)}")

Αρχικές εγγραφές us_shots: 605446
Overlapping matches: 1576
Εγγραφές μετά την αφαίρεση: 566725


Drop 'match_id' from DataFrames

In [8]:
us_shots.drop(['match_id'], axis=1, inplace=True)
sb_shots.drop(['match_id'], axis=1, inplace=True)

print(f"{list(us_shots)} και {list(sb_shots)}")

['x_loc', 'y_loc', 'distance_to_goal', 'angle_to_goal', 'is_goal'] και ['x_loc', 'y_loc', 'distance_to_goal', 'angle_to_goal', 'is_goal']


Merge 2 dataframes

In [9]:
shots = pd.concat([us_shots, sb_shots], ignore_index=True)
print(f"Μέγεθος Understat: {len(us_shots)}")
print(f"Μέγεθος StatsBomb: {len(sb_shots)}")
print(f"Μέγεθος Συνολικό : {len(shots)}")

Μέγεθος Understat: 566725
Μέγεθος StatsBomb: 55857
Μέγεθος Συνολικό : 622582


In [52]:
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss
import numpy as np

all_results = []

def evaluate_model(y_true, y_pred, model_name="Model"):
    loss = log_loss(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    brier = brier_score_loss(y_true, y_pred)
    actual_goals = int(np.sum(y_true))
    expected_goals = float(np.sum(y_pred))

    print(f"╔{'═'*45}╗")
    print(f"  Αποτελέσματα για: {model_name}")
    print(f"╠{'═'*45}╣")
    print(f"  Log Loss:        {loss:.4f}")
    print(f"  ROC AUC:         {auc:.4f}")
    print(f"  Brier Score:     {brier:.4f}")
    print(f"  Actual Goals:    {actual_goals}")
    print(f"  Expected Goals:  {expected_goals:.1f}")
    print(f"╚{'═'*45}╝\n")

    return {
        "Model": model_name,
        "Log Loss": round(loss, 4),
        "ROC AUC": round(auc, 4),
        "Brier Score": round(brier, 4),
        "Actual Goals": actual_goals,
        "Expected Goals": round(expected_goals, 1)
    }

In [69]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold, cross_val_score
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import BorderlineSMOTE
import optuna
from optuna.samplers import TPESampler

X = shots.drop(columns=['is_goal'])
y = shots['is_goal'].astype(np.float32)

scale_pos_weight_value = (y == 0).sum() / (y == 1).sum()

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
skf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [54]:
import torch
import torch.nn as nn
from skorch.callbacks import EarlyStopping
from skorch import NeuralNetBinaryClassifier

device = 'cuda'

# in_features: αριθμός τον εισαγώμενων eatures
# 1st hidden layer: 128 (default, αν το μεγαλώσουμε, κάνει overfitting)
# 2nd hidden layer: 64 (συμπιέζουμε στους μισούς νευρώνες)
# dropout=0.3: 30% πιθανότητα απενεργοποίησης
# τα παραπάνω τα θεωρούμε αυθαίρετα ως default και θα αποτελούν το baseline default

class xG_MLP(nn.Module):
    def __init__(self, in_features, dropout=0.3):
        super().__init__()

        self.fc1 = nn.Linear(in_features, 128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.out = nn.Linear(64, 1)
    def forward(self, x):
        x = x.float()
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        x = self.out(x)
        return x.squeeze(-1)

# Return το item ως float
class SafeBCE(nn.BCEWithLogitsLoss):
    def forward(self, input, target):
        return super().forward(input, target.float())

BL6.1.1.1 — Default Params + Imbalance: None + Calibration: None

In [55]:
torch.manual_seed(42)

bl6_1_1_1 = make_imb_pipeline(
    StandardScaler(),
    NeuralNetBinaryClassifier(
        module=xG_MLP,
        module__in_features=X.shape[1],
        criterion=SafeBCE,
        optimizer=torch.optim.Adam,
        lr=0.001,
        batch_size=1024,
        max_epochs=200,
        callbacks=[EarlyStopping(monitor='valid_loss', patience=10)],
        device=device,
        verbose=1
    )
)

predictions_bl6_1_1_1 = cross_val_predict(
    bl6_1_1_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl6_1_1_1, model_name="BL6.1.1.1")
all_results.append(res)

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.2975       0.9052        0.2822  4.7766
      2        0.2860       0.9053        0.2817  4.8195
      3        0.2851       0.9052        0.2813  4.8176
      4        0.2846       0.9052        0.2813  4.8386
      5        0.2842       0.9052        0.2811  4.7563
      6        0.2840       0.9052        0.2811  4.8040
      7        0.2834       0.9052        0.2810  4.9999
      8        0.2835       0.9052        0.2809  4.8144
      9        0.2831       0.9052        0.2809  5.3369
     10        0.2829       0.9052        0.2808  4.7621
     11        0.2829       0.9052        0.2808  4.8110
     12        0.2827       0.9052        0.2807  4.5747
     13        0.2826       0.9052        0.2807  4.6500
     14        0.2825       0.9052        0.2807  4.6913
     15        0.2825       0.9052        0.2807  4.6512
     16        0.2821       0.9

BL6.1.1.2 — Default Params + Imbalance: None + Calibration: Platt Scaling

In [56]:
P_raw = predictions_bl6_1_1_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(C=np.inf, solver='lbfgs', random_state=42)

predictions_bl6_1_1_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl6_1_1_2, model_name="BL6.1.1.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL6.1.1.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2846
  ROC AUC:         0.7437
  Brier Score:     0.0798
  Actual Goals:    60584
  Expected Goals:  60589.1
╚═════════════════════════════════════════════╝



BL6.1.1.3 — Default Params + Imbalance: None + Calibration: Isotonic Regression

In [57]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl5_1_1_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl5_1_1_3, model_name="BL5.1.1.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.1.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2800
  ROC AUC:         0.7432
  Brier Score:     0.0790
  Actual Goals:    60584
  Expected Goals:  60589.9
╚═════════════════════════════════════════════╝



BL6.1.2.1 — Default Params + Imbalance: Class Weight + Calibration: None

In [58]:
pos_weight_tensor = torch.tensor([scale_pos_weight_value], dtype=torch.float32, device=device)

torch.manual_seed(42)

bl6_1_2_1 = make_imb_pipeline(
    StandardScaler(),
    NeuralNetBinaryClassifier(
        module=xG_MLP,
        module__in_features=X.shape[1],
        criterion=SafeBCE,
        criterion__pos_weight=pos_weight_tensor,
        optimizer=torch.optim.Adam,
        lr=0.001,
        batch_size=1024,
        iterator_train__pin_memory=True,
        max_epochs=200,
        callbacks=[EarlyStopping(monitor='valid_loss', patience=10)],
        device=device,
        verbose=0
    )
)

predictions_bl6_1_2_1 = cross_val_predict(
    bl6_1_2_1,
    X,
    y,
    cv=skf,
    method='predict_proba',
    verbose=1
)[:, 1]

res = evaluate_model(y, predictions_bl6_1_2_1, model_name="BL6.1.2.1")
all_results.append(res)

[Parallel(n_jobs=1)]: Done  10 out of  10 | elapsed: 18.5min finished


╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL6.1.2.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5529
  ROC AUC:         0.7441
  Brier Score:     0.1883
  Actual Goals:    60584
  Expected Goals:  249339.1
╚═════════════════════════════════════════════╝



BL6.1.2.2 — Default Params + Imbalance: Class Weight + Calibration: Platt Scaling

In [59]:
P_raw = predictions_bl6_1_2_1.reshape(-1, 1)

predictions_bl6_1_2_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl6_1_2_2, model_name="BL6.1.2.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL6.1.2.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2810
  ROC AUC:         0.7440
  Brier Score:     0.0793
  Actual Goals:    60584
  Expected Goals:  60589.3
╚═════════════════════════════════════════════╝



BL6.1.2.3 — Default Params + Imbalance: Class Weight + Calibration: Isotonic Regression

In [60]:
predictions_bl6_1_2_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl6_1_2_3, model_name="BL6.1.2.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL6.1.2.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2799
  ROC AUC:         0.7434
  Brier Score:     0.0790
  Actual Goals:    60584
  Expected Goals:  60587.0
╚═════════════════════════════════════════════╝



BL6.1.3.1 — Default Params + Imbalance: Undersampling + Calibration: None

In [61]:
torch.manual_seed(42)

bl6_1_3_1 = make_imb_pipeline(
    StandardScaler(),
    RandomUnderSampler(random_state=42),
    NeuralNetBinaryClassifier(
        module=xG_MLP,
        module__in_features=X.shape[1],
        criterion=SafeBCE,
        optimizer=torch.optim.Adam,
        lr=0.001,
        batch_size=1024,
        iterator_train__pin_memory=True,
        max_epochs=200,
        callbacks=[EarlyStopping(monitor='valid_loss', patience=10)],
        device=device,
        verbose=1
    )
)

predictions_bl6_1_3_1 = cross_val_predict(
    bl6_1_3_1,
    X,
    y,
    cv=skf,
    method='predict_proba',
    verbose=1
)[:, 1]

res = evaluate_model(y, predictions_bl6_1_3_1, model_name="BL6.1.3.1")
all_results.append(res)

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.9730       0.5000        0.6739  0.8702
      2        0.7035       0.6741        0.6306  0.8633
      3        0.6687       0.6782        0.6217  0.9868
      4        0.6625       0.6779        0.6174  0.8590
      5        0.6576       0.6765        0.6156  0.8567
      6        0.6539       0.6770        0.6142  0.8806
      7        0.6528       0.6749        0.6132  0.8666
      8        0.6501       0.6748        0.6134  0.8625
      9        0.6497       0.6741        0.6123  0.8642
     10        0.6484       0.6738        0.6120  0.8743
     11        0.6481       0.6734        0.6116  0.8665
     12        0.6465       0.6727        0.6113  0.8660
     13        0.6465       0.6719        0.6111  0.8706
     14        0.6459       0.6719        0.6108  0.8576
     15        0.6459       0.6710        0.6105  1.0008
     16        0.6445       0.6

[Parallel(n_jobs=1)]: Done  10 out of  10 | elapsed: 20.6min finished


╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL6.1.3.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.6603
  ROC AUC:         0.7429
  Brier Score:     0.2298
  Actual Goals:    60584
  Expected Goals:  296409.0
╚═════════════════════════════════════════════╝



BL6.1.3.2 — Default Params + Imbalance: Undersampling + Calibration: Platt Scaling

In [62]:
P_raw = predictions_bl6_1_3_1.reshape(-1, 1)

predictions_bl6_1_3_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl6_1_3_2, model_name="BL6.1.3.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL6.1.3.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2810
  ROC AUC:         0.7428
  Brier Score:     0.0793
  Actual Goals:    60584
  Expected Goals:  60591.7
╚═════════════════════════════════════════════╝



BL6.1.3.3 — Default Params + Imbalance: Undersampling + Calibration: Isotonic Regression

In [63]:
predictions_bl6_1_3_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl6_1_3_3, model_name="BL6.1.3.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL6.1.3.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2803
  ROC AUC:         0.7421
  Brier Score:     0.0790
  Actual Goals:    60584
  Expected Goals:  60511.0
╚═════════════════════════════════════════════╝



BL6.1.4.1 — Default Params + Imbalance: Oversampling + Calibration: None

In [49]:
torch.manual_seed(42)

bl6_1_4_1 = make_imb_pipeline(
    StandardScaler(),
    BorderlineSMOTE(random_state=42),
    NeuralNetBinaryClassifier(
        module=xG_MLP,
        module__in_features=X.shape[1],
        criterion=SafeBCE,
        optimizer=torch.optim.Adam,
        lr=0.001,
        batch_size=4096,
        iterator_train__pin_memory=True,
        max_epochs=200,
        callbacks=[EarlyStopping(monitor='valid_loss', patience=10)],
        device=device,
        verbose=1
    )
)

predictions_bl6_1_4_1 = cross_val_predict(
    bl6_1_4_1,
    X,
    y,
    cv=skf,
    method='predict_proba',
    verbose=1
)[:, 1]

res = evaluate_model(y, predictions_bl6_1_4_1, model_name="BL6.1.4.1")
all_results.append(res)

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.5311       0.5000        2.7157  6.9370
      2        0.9579       0.6164        0.6806  6.8897
      3        0.6501       0.6322        0.6920  6.9661
      4        0.6485       0.6436        0.6818  6.6999
      5        0.6442       0.6497        0.6757  6.5520
      6        0.6427       0.6553        0.6709  6.7172
      7        0.6410       0.6587        0.6682  6.7016
      8        0.6396       0.6572        0.6706  6.6707
      9        0.6398       0.6610        0.6661  6.6979
     10        0.6387       0.6614        0.6658  6.8814
     11        0.6384       0.6616        0.6651  6.6739
     12        0.6377       0.6197        0.7181  6.4339
     13        0.6480       0.6615        0.6611  6.6905
     14        0.6367       0.6608        0.6625  6.5347
     15        0.6373       0.6609        0.6611  6.4385
     16        0.6369       0.6

[Parallel(n_jobs=1)]: Done  10 out of  10 | elapsed: 51.3min finished


╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL6.1.4.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.8388
  ROC AUC:         0.7424
  Brier Score:     0.3050
  Actual Goals:    60584
  Expected Goals:  351366.0
╚═════════════════════════════════════════════╝



BL6.1.4.2 — Default Params + Imbalance: Oversampling + Calibration: Platt Scaling

In [64]:
P_raw = predictions_bl6_1_4_1.reshape(-1, 1)

predictions_bl6_1_4_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl6_1_4_2, model_name="BL6.1.4.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL6.1.4.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2831
  ROC AUC:         0.7424
  Brier Score:     0.0800
  Actual Goals:    60584
  Expected Goals:  60583.2
╚═════════════════════════════════════════════╝



BL6.1.4.3 — Default Params + Imbalance: Oversampling + Calibration: Isotonic Regression

In [65]:
predictions_bl6_1_4_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl6_1_4_3, model_name="BL6.1.4.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL6.1.4.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2808
  ROC AUC:         0.7416
  Brier Score:     0.0792
  Actual Goals:    60584
  Expected Goals:  60587.0
╚═════════════════════════════════════════════╝



BL6.2.1.1 — HyperParams + Imbalance: None + Calibration: None

In [71]:
class xG_MLP(nn.Module):
    def __init__(self, in_features, hidden_layer_sizes=(128, 64), dropout=0.3, use_batch_norm=False):
        super().__init__()
        layers = []
        prev_dim = in_features
        for hidden_dim in hidden_layer_sizes:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        x = x.float()
        return self.network(x).squeeze(-1)

In [75]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_mlp(trial):
    arch_map = {
        '64_32': (64, 32),
        '128_64': (128, 64),
        '256_128': (256, 128),
        '256_128_64': (256, 128, 64)
    }
    arch_key = trial.suggest_categorical('hidden_arch', list(arch_map.keys()))
    hidden_layers = arch_map[arch_key]

    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
    use_batch_norm = trial.suggest_categorical('use_batch_norm', [True, False])
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [512, 1024, 2048, 4096])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    torch.manual_seed(42)
    model = make_imb_pipeline(
        StandardScaler(),
        NeuralNetBinaryClassifier(
            module=xG_MLP,
            module__in_features=X.shape[1],
            module__hidden_layer_sizes=hidden_layers,
            module__dropout=dropout,
            module__use_batch_norm=use_batch_norm,
            criterion=SafeBCE,
            optimizer=torch.optim.Adam,
            optimizer__weight_decay=weight_decay,
            lr=lr,
            batch_size=batch_size,
            iterator_train__pin_memory=True,
            max_epochs=200,
            callbacks=[EarlyStopping(monitor='valid_loss', patience=10)],
            device=device,
            verbose=1
        )
    )

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()

sampler = optuna.samplers.TPESampler(seed=42)
study_mlp = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για MLP Neural Network...")
study_mlp.optimize(objective_mlp, n_trials=20, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_mlp.best_params)
print(f"Καλύτερο CV Log Loss: {study_mlp.best_value:.5f}")

Έναρξη Optuna Tuning για MLP Neural Network...


  0%|          | 0/20 [00:00<?, ?it/s]

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.2962       0.9052        0.2839  5.7123
      2        0.2836       0.9053        0.2834  5.7161
      3        0.2829       0.9051        0.2828  5.6487
      4        0.2830       0.9052        0.2818  5.6739
      5        0.2830       0.9052        0.2815  5.8029
      6        0.2827       0.9053        0.2816  5.6713
      7        0.2824       0.9053        0.2814  5.6667
      8        0.2822       0.9049        0.2813  5.6817
      9        0.2818       0.9052        0.2818  5.6750
     10        0.2818       0.9053        0.2811  5.6600
     11        0.2817       0.9052        0.2830  5.7062
     12        0.2818       0.9053        0.2830  5.6612
     13        0.2819       0.9049        0.2823  5.6440
     14        0.2819       0.9052        0.2822  5.6590
     15        0.2819       0.9049        0.2823  5.6628
     16        0.2819       0.9

Best trial: 0. Best value: 0.281623:   5%|▌         | 1/20 [10:43<3:23:45, 643.46s/it]

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.3369       0.9052        0.2819  6.8468
      2        0.2877       0.9054        0.2812  6.8310
      3        0.2867       0.9053        0.2811  6.8274
      4        0.2862       0.9054        0.2812  6.8041
      5        0.2859       0.9054        0.2812  6.6730
      6        0.2855       0.9054        0.2812  6.8240
      7        0.2853       0.9054        0.2812  6.8357
      8        0.2850       0.9054        0.2811  6.8558
      9        0.2846       0.9054        0.2813  6.8441
     10        0.2846       0.9054        0.2814  6.8669
     11        0.2846       0.9054        0.2815  6.8354
     12        0.2844       0.9055        0.2813  6.8495
Stopping since valid_loss has not improved in the last 10 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.3232 

...

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.3293       0.9053        0.2848  6.1810
      2        0.2861       0.9053        0.2826  6.2205
      3        0.2848       0.9054        0.2821  6.3183
      4        0.2843       0.9054        0.2819  6.2388
      5        0.2843       0.9054        0.2818  6.4084
      6        0.2838       0.9054        0.2817  6.1974
      7        0.2838       0.9054        0.2816  6.3845
      8        0.2835       0.9054        0.2814  6.2806
      9        0.2834       0.9054        0.2814  6.3792
     10        0.2832       0.9053        0.2814  6.2655
     11        0.2830       0.9053        0.2813  6.2329
     12        0.2829       0.9052        0.2812  6.3691
     13        0.2829       0.9053        0.2812  6.2308
     14        0.2827       0.9052        0.2812  6.3545
     15        0.2826       0.9052        0.2811  6.2378
     16        0.2826       0.9

Best trial: 6. Best value: 0.280425:  95%|█████████▌| 19/20 [5:22:13<17:41, 1061.12s/it]

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.2875       0.9053        0.2820  6.4037
      2        0.2831       0.9053        0.2814  6.6730
      3        0.2826       0.9053        0.2815  6.4641
      4        0.2822       0.9053        0.2816  6.3761
      5        0.2822       0.9053        0.2816  6.3983
      6        0.2821       0.9053        0.2815  6.4808
      7        0.2820       0.9053        0.2815  6.4109
      8        0.2820       0.9053        0.2815  6.4934
      9        0.2819       0.9052        0.2815  6.4898
     10        0.2818       0.9053        0.2813  6.4904
     11        0.2817       0.9053        0.2814  6.5894
     12        0.2817       0.9052        0.2814  6.3512
     13        0.2817       0.9053        0.2814  6.5117
     14        0.2816       0.9053        0.2814  6.3752
     15        0.2817       0.9052        0.2812  6.5574
     16        0.2815       0.9

Best trial: 6. Best value: 0.280425: 100%|██████████| 20/20 [5:41:45<00:00, 1025.26s/it]


Βέλτιστες Παράμετροι: {'hidden_arch': '256_128', 'dropout': 0.2, 'use_batch_norm': False, 'lr': 0.0004470608546778491, 'batch_size': 2048, 'weight_decay': 1.7345566642360933e-05}
Καλύτερο CV Log Loss: 0.28043


In [ ]:
best_params = {'hidden_arch': '256_128', 'dropout': 0.2, 'use_batch_norm': False, 'lr': 0.0004470608546778491, 'batch_size': 2048, 'weight_decay': 1.7345566642360933e-05}

In [ ]:
best = study_mlp.best_params
arch_map = {
    '64_32': (64, 32),
    '128_64': (128, 64),
    '256_128': (256, 128),
    '256_128_64': (256, 128, 64)
}

if 'hidden_layer_sizes' in best:
    hidden_layers = best['hidden_layer_sizes']
else:
    hidden_layers = arch_map[best['hidden_arch']]

torch.manual_seed(42)

bl6_2_1_1 = make_imb_pipeline(
    StandardScaler(),
    NeuralNetBinaryClassifier(
        module=xG_MLP,
        module__in_features=X.shape[1],
        module__hidden_layer_sizes=hidden_layers,
        module__dropout=best['dropout'],
        module__use_batch_norm=best['use_batch_norm'],
        criterion=SafeBCE,
        optimizer=torch.optim.Adam,
        optimizer__weight_decay=best['weight_decay'],
        lr=best['lr'],
        batch_size=best['batch_size'],
        iterator_train__pin_memory=True,
        max_epochs=200,
        callbacks=[EarlyStopping(monitor='valid_loss', patience=10)],
        device=device,
        verbose=0
    )
)

predictions_bl6_2_1_1 = cross_val_predict(
    bl6_2_1_1,
    X,
    y,
    cv=skf,
    method='predict_proba',
    verbose=1
)[:, 1]

res = evaluate_model(y, predictions_bl6_2_1_1, model_name="BL6.2.1.1")
all_results.append(res)

BL6.2.2.1 — HyperParams + Imbalance: Class Weight + Calibration: None

In [77]:
def objective_cw(trial):
    arch_map = {
        '64_32': (64, 32),
        '128_64': (128, 64),
        '256_128': (256, 128),
        '256_128_64': (256, 128, 64)
    }
    arch_key = trial.suggest_categorical('hidden_arch', list(arch_map.keys()))
    hidden_layers = arch_map[arch_key]

    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
    use_batch_norm = trial.suggest_categorical('use_batch_norm', [True, False])
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [512, 1024, 2048, 4096])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    torch.manual_seed(42)
    model = make_imb_pipeline(
        StandardScaler(),
        NeuralNetBinaryClassifier(
            module=xG_MLP,
            module__in_features=X.shape[1],
            module__hidden_layer_sizes=hidden_layers,
            module__dropout=dropout,
            module__use_batch_norm=use_batch_norm,
            criterion=SafeBCE,
            criterion__pos_weight=pos_weight_tensor,
            optimizer=torch.optim.Adam,
            optimizer__weight_decay=weight_decay,
            lr=lr,
            batch_size=batch_size,
            iterator_train__pin_memory=True,
            max_epochs=200,
            callbacks=[EarlyStopping(monitor='valid_loss', patience=10)],
            device=device,
            verbose=0
        )
    )
    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_cw = optuna.create_study(direction='minimize', sampler=sampler)
print("Έναρξη Optuna Tuning για MLP με Class Weighting...")

study_cw.optimize(objective_cw, n_trials=20, show_progress_bar=True)
print("\nΒέλτιστες Παράμετροι:", study_cw.best_params)
print(f"Καλύτερο CV Log Loss: {study_cw.best_value:.5f}")

Έναρξη Optuna Tuning για MLP με Class Weighting...


Best trial: 5. Best value: 0.540319: 100%|██████████| 20/20 [7:12:57<00:00, 1298.85s/it]  


Βέλτιστες Παράμετροι: {'hidden_arch': '256_128', 'dropout': 0.5, 'use_batch_norm': True, 'lr': 0.00010257563974185662, 'batch_size': 512, 'weight_decay': 1.4063366777718176e-05}
Καλύτερο CV Log Loss: 0.54032


In [ ]:
best_params = {'hidden_arch': '256_128', 'dropout': 0.5, 'use_batch_norm': True, 'lr': 0.00010257563974185662, 'batch_size': 512, 'weight_decay': 1.4063366777718176e-05}

BL6.2.3.1 — HyperParams + Imbalance: Undersampling + Calibration: None

In [79]:
def objective_us(trial):
    arch_map = {
        '64_32': (64, 32),
        '128_64': (128, 64),
        '256_128': (256, 128),
        '256_128_64': (256, 128, 64)
    }
    arch_key = trial.suggest_categorical('hidden_arch', list(arch_map.keys()))
    hidden_layers = arch_map[arch_key]

    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
    use_batch_norm = trial.suggest_categorical('use_batch_norm', [True, False])
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [512, 1024, 2048, 4096])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    torch.manual_seed(42)
    model = make_imb_pipeline(
        StandardScaler(),
        RandomUnderSampler(random_state=42),
        NeuralNetBinaryClassifier(
            module=xG_MLP,
            module__in_features=X.shape[1],
            module__hidden_layer_sizes=hidden_layers,
            module__dropout=dropout,
            module__use_batch_norm=use_batch_norm,
            criterion=SafeBCE,
            optimizer=torch.optim.Adam,
            optimizer__weight_decay=weight_decay,
            lr=lr,
            batch_size=batch_size,
            iterator_train__pin_memory=True,
            max_epochs=200,
            callbacks=[EarlyStopping(monitor='valid_loss', patience=10)],
            device=device,
            verbose=0
        )
    )
    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_us = optuna.create_study(direction='minimize', sampler=sampler)
print("Έναρξη Optuna Tuning για MLP με Undersampling...")

study_us.optimize(objective_us, n_trials=20, show_progress_bar=True)
print("\nΒέλτιστες Παράμετροι:", study_us.best_params)
print(f"Καλύτερο CV Log Loss: {study_us.best_value:.5f}")

Έναρξη Optuna Tuning για MLP με Undersampling...


Best trial: 17. Best value: 0.59314: 100%|██████████| 20/20 [2:50:13<00:00, 510.70s/it]   


Βέλτιστες Παράμετροι: {'hidden_arch': '256_128_64', 'dropout': 0.1, 'use_batch_norm': False, 'lr': 0.000771526560647033, 'batch_size': 4096, 'weight_decay': 1.008600292268101e-05}
Καλύτερο CV Log Loss: 0.59314


In [81]:
def objective_os(trial):
    arch_map = {
        '64_32': (64, 32),
        '128_64': (128, 64),
        '256_128': (256, 128),
        '256_128_64': (256, 128, 64)
    }
    arch_key = trial.suggest_categorical('hidden_arch', list(arch_map.keys()))
    hidden_layers = arch_map[arch_key]

    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
    use_batch_norm = trial.suggest_categorical('use_batch_norm', [True, False])
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [512, 1024, 2048, 4096])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    torch.manual_seed(42)
    model = make_imb_pipeline(
        StandardScaler(),
        BorderlineSMOTE(random_state=42),
        NeuralNetBinaryClassifier(
            module=xG_MLP,
            module__in_features=X.shape[1],
            module__hidden_layer_sizes=hidden_layers,
            module__dropout=dropout,
            module__use_batch_norm=use_batch_norm,
            criterion=SafeBCE,
            optimizer=torch.optim.Adam,
            optimizer__weight_decay=weight_decay,
            lr=lr,
            batch_size=batch_size,
            iterator_train__pin_memory=True,
            max_epochs=200,
            callbacks=[EarlyStopping(monitor='valid_loss', patience=10)],
            device=device,
            verbose=0
        )
    )
    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_os = optuna.create_study(direction='minimize', sampler=sampler)
print("Έναρξη Optuna Tuning για MLP με Oversampling...")

study_os.optimize(objective_os, n_trials=20, show_progress_bar=True)
print("\nΒέλτιστες Παράμετροι:", study_os.best_params)
print(f"Καλύτερο CV Log Loss: {study_os.best_value:.5f}")

Έναρξη Optuna Tuning για MLP με Oversampling...


Best trial: 3. Best value: 0.831023: 100%|██████████| 20/20 [8:49:52<00:00, 1589.62s/it]  


Βέλτιστες Παράμετροι: {'hidden_arch': '256_128', 'dropout': 0.5, 'use_batch_norm': False, 'lr': 0.0004201672054372534, 'batch_size': 4096, 'weight_decay': 0.0003550304858128307}
Καλύτερο CV Log Loss: 0.83102
